## 查看是否开仓的盈亏详情

In [4]:
import pandas as pd

def analyze_trades_aligned(
    trades_path: str,
    all_trades_path: str,
    pred_threshold: float = 0.1
):
    """
    对齐键：
      - trades_ledger: (code, strategy, entry_date)
      - all_signals_trades: (code, strategy, signal_date)  ——> 在对齐时临时改名为 entry_date

    统计内容：
      1) 在 predicted_return > 阈值 下：
         - trades_ledger 与 all_signals_trades 的记录数、pnl均值、盈利数与占比（整体 + 按 strategy）
      2) 交集/差集：
         - 相同记录数
         - 差集 = all_signals_trades - trades_ledger（同样已按阈值过滤 + 日期键对齐）
         - 对差集做 pnl均值、盈利数/占比（整体 + 按 strategy）
    返回：一个 dict，包含筛选后的数据集与差集，便于你继续做图/导出
    """

    # ------- 读取 -------
    trades = pd.read_csv(trades_path, parse_dates=['entry_date'])
    all_trades = pd.read_csv(all_trades_path, parse_dates=['entry_date','signal_date'])

    # 列名统一（若没有 code，用 symbol 代替）
    for df in [trades, all_trades]:
        if 'code' not in df.columns and 'symbol' in df.columns:
            df['code'] = df['symbol'].astype(str)
        else:
            df['code'] = df['code'].astype(str)

    # ------- 基础过滤：predicted_return > 阈值 -------
    t_sel = trades.loc[trades['predicted_return'] > pred_threshold].copy()
    a_sel = all_trades.loc[all_trades['predicted_return'] > pred_threshold].copy()

    # ------- 输出函数：整体 + 按 strategy -------
    def summarize(df: pd.DataFrame, name: str):
        if df.empty:
            print(f"\n[{name}] 无记录（predicted_return > {pred_threshold}）")
            return
        total = len(df)
        avg_pnl = df['pnl_pct'].mean()
        win_count = (df['pnl_pct'] > 0).sum()
        win_rate = win_count / total if total > 0 else 0.0
        print(f"\n[{name}] 总数={total}, pnl均值={avg_pnl:.4f}, 盈利数={win_count}, 盈利占比={win_rate:.2%}")
        by_strat = df.groupby('strategy')['pnl_pct'].agg(
            count='count',
            avg_pnl='mean',
            win_count=lambda s: (s > 0).sum(),
            win_rate=lambda s: (s > 0).mean()
        )
        print("按 strategy：")
        print(by_strat)

    # ------- 1) 分别统计 trades_ledger / all_signals_trades -------
    summarize(t_sel, "Trades Ledger")
    summarize(a_sel, "All Signals Trades")

    # ------- 2) 交集/差集（对齐键：ledger.entry_date ↔ all.signal_date）-------
    # 先构造对齐用的 Key DataFrame
    ledger_keys = t_sel[['code', 'strategy', 'entry_date']].drop_duplicates()
    all_keys = a_sel[['code', 'strategy', 'signal_date']].drop_duplicates() \
                     .rename(columns={'signal_date': 'entry_date'})

    # 交集与差集（注意此处已是阈值过滤后的集合）
    inter_keys = pd.merge(ledger_keys, all_keys, on=['code','strategy','entry_date'], how='inner')
    # 差集：在 All 中但不在 Ledger 中
    diff_keys = pd.merge(
        all_keys, ledger_keys,
        on=['code','strategy','entry_date'], how='left', indicator=True
    )
    diff_keys = diff_keys[diff_keys['_merge'] == 'left_only'] \
                       .drop(columns=['_merge'])

    print(f"\n相同记录数: {len(inter_keys)}, 不同记录数(候选未开仓): {len(diff_keys)}")

    # 用差集 key 去 a_sel 里取完整行（注意把 entry_date -> signal_date 再对齐回去）
    if not diff_keys.empty:
        # 为 merge 回 a_sel，改回 signal_date
        diff_keys_for_join = diff_keys.rename(columns={'entry_date': 'signal_date'})
        diff_trades = a_sel.merge(
            diff_keys_for_join, on=['code','strategy','signal_date'], how='inner'
        )
    else:
        diff_trades = a_sel.iloc[0:0].copy()  # 空集

    # 对差集做统计
    summarize(diff_trades, "差集 All_only（候选未开仓）")

    return {
        "trades_filtered": t_sel,       # trades_ledger 中 predicted_return > 阈值 的样本
        "all_trades_filtered": a_sel,   # all_signals_trades 中 predicted_return > 阈值 的样本
        "inter_keys": inter_keys,       # 键交集（对齐后）
        "diff_trades": diff_trades      # 差集的完整记录（已按阈值过滤）
    }

# 示例：
analyze_trades_aligned(
     "trades_ledger.csv",
     "all_signals_trades.csv",
     pred_threshold=0.1)


[Trades Ledger] 总数=2221, pnl均值=0.1781, 盈利数=1579, 盈利占比=71.09%
按 strategy：
          count   avg_pnl  win_count  win_rate
strategy                                      
S1          137  0.621153        127  0.927007
S2          783  0.301019        556  0.710089
S3         1301  0.057517        896  0.688701

[All Signals Trades] 总数=3767, pnl均值=0.1194, 盈利数=2391, 盈利占比=63.47%
按 strategy：
          count   avg_pnl  win_count  win_rate
strategy                                      
S1          196  0.357409        114  0.581633
S2         1083  0.254074        701  0.647276
S3         2488  0.041979       1576  0.633441

相同记录数: 2221, 不同记录数(候选未开仓): 1546

[差集 All_only（候选未开仓）] 总数=1546, pnl均值=0.0497, 盈利数=853, 盈利占比=55.17%
按 strategy：
          count   avg_pnl  win_count  win_rate
strategy                                      
S1           59  0.131617         28  0.474576
S2          300  0.131547        145  0.483333
S3         1187  0.024948        680  0.572873


{'trades_filtered':         symbol strategy entry_date  entry_price   exit_date  exit_price  \
 91           4       S1 2024-10-10    14.810000  2024-10-31   20.820000   
 638         23       S3 2024-06-07     1.720000  2024-06-26    2.020000   
 816         29       S2 2024-08-30    11.060000  2024-11-18   16.570000   
 821         29       S1 2025-04-21    15.050000  2025-06-17   19.800000   
 822         29       S2 2025-08-07    20.790000  2025-09-10   27.180000   
 ...        ...      ...        ...          ...         ...         ...   
 178548  688800       S3 2021-09-10    41.316214  2021-09-30   36.524913   
 178549  688800       S3 2022-02-08    55.668559  2022-02-25   60.632325   
 178550  688800       S3 2022-03-15    49.583768  2022-03-18   56.590170   
 178587  688981       S3 2020-12-22    53.990000  2021-01-04   58.370000   
 178588  688981       S3 2021-03-25    53.250000  2021-04-02   57.210000   
 
          pnl_pct  entry_pos               exit_reason  initial_sto